<a href="https://colab.research.google.com/github/prasertrak/Advanced-Data-Engineering-and-Applied-Analytics/blob/main/Lab1_data_cleaning_validation_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Cleaning & Validating Data with Python
### Order Fulfillment Analytics Pipeline / Cleaning the dirty_order.csv


### Learning Objectives
- Detect dirty data
- Clean invalid records
- Handle NULL values
- Standardize text
- Quarantine bad records
- Perform reconciliation


In [ ]:
import pandas as pd

## Step 1. Load dirty_orders.csv into pandas

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving dirty_orders.csv to dirty_orders.csv


In [ ]:
df = pd.read_csv("dirty_orders.csv")
raw_dirty_df = df.copy()


In [ ]:
print("Before cleaning:", df.shape)
df.head()

Before cleaning: (21, 8)


,order_id,customer_id,order_date,product_category,quantity,unit_price,delivery_days,status
0,O1001,C001,2026-01-03,Electronics,1,15900.00,3.0,Failed
1,O1002,C002,2026/01/05,office supplies,4,120.00,8.0,Failed
2,O1003,C003,06-01-2026,FURNITURE,1,8900.00,7.0,Succeeded
3,O1004,C001,2026-01-10,Office Supplies,2,350.00,10.0,Succeeded
4,O1005,c004,2026-01-12,electronics,2,1290.00,5.0,Succeeded


In [ ]:
df

,order_id,customer_id,order_date,product_category,quantity,unit_price,delivery_days,status
0,O1001,C001,2026-01-03,Electronics,1,15900.00,3.0,Failed
1,O1002,C002,2026/01/05,office supplies,4,120.00,8.0,Failed
2,O1003,C003,06-01-2026,FURNITURE,1,8900.00,7.0,Succeeded
3,O1004,C001,2026-01-10,Office Supplies,2,350.00,10.0,Succeeded
4,O1005,c004,2026-01-12,electronics,2,1290.00,5.0,Succeeded
5,O1006,C005,2026-01-14,Furniture,1,4500.00,NaN,Failed
6,O1007,C006,2026-01-18,Office Supplies,10,45.00,2.0,Succeeded
7,O1007,C006,2026-01-18,Office Supplies,10,45.00,2.0,Succeeded
8,O1008,C007,2026-01-21,Electronics,1,"25,900.00",12.0,Refunded
9,O1009,C008,2026-01-23,Furniture,two,3200.00,6.0,Succeeded


## 1. ลบแถวซ้ำ

In [ ]:
df = df.drop_duplicates()

# 2. ลบช่องว่างส่วนเกิน


In [ ]:
# ค้นหาคอลัมน์ที่เป็นข้อความ
text_columns = df.select_dtypes(include="object").columns

# ลบช่องว่างด้านหน้าและด้านหลังข้อความ
# เช่น " C001 " -> "C001"
# และ " furniture " -> "furniture"
for column in text_columns:
    df[column] = df[column].str.strip()

/tmp/ipykernel_5282/283403190.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].str.strip()


# 3. ปรับตัวพิมพ์เล็ก-ใหญ่ให้เป็นมาตรฐานเดียวกัน


In [ ]:
# รหัสลูกค้าใช้ตัวพิมพ์ใหญ่เสมอ
# เช่น "c004" -> "C004"
df["customer_id"] = df["customer_id"].str.upper()

# ชื่อหมวดหมู่สินค้าใช้รูปแบบ Title Case
# เช่น "FURNITURE" -> "Furniture"
# และ "Office supplies" -> "Office Supplies"
df["product_category"] = df["product_category"].str.title()

/tmp/ipykernel_5282/3947668212.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["customer_id"] = df["customer_id"].str.upper()
/tmp/ipykernel_5282/3947668212.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["product_category"] = df["product_category"].str.title()


# 4. ปรับวันที่ให้เป็นรูปแบบเดียวกัน


In [ ]:
# รองรับกรณีที่วันที่ต้นทางมีหลายรูปแบบ
# เช่น "2026/01/05" และ "06-01-2026"
#
# dayfirst=True: หากรูปแบบกำกวม ให้ถือว่าตัวเลขแรกเป็นวัน
# errors="coerce": หากแปลงไม่ได้ เช่น "invalid_date" ให้เปลี่ยนเป็น NaT
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# หมายเหตุ:
# NaT คือค่าว่างสำหรับข้อมูลประเภทวันที่
# ยังไม่ต้องแปลงกลับเป็นข้อความ เพราะ datetime เหมาะกับการวิเคราะห์มากกว่า

/tmp/ipykernel_5282/3407696415.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["order_date"] = pd.to_datetime(


# 5. ปรับค่าตัวเลข


In [ ]:
# แปลงคำว่า "two" เป็นเลข 2 ก่อนแปลงชนิดข้อมูล
# เหมาะกับกรณีที่ความหมายชัดเจนและแก้ไขได้อย่างมั่นใจ
df["quantity"] = df["quantity"].replace({
    "two": 2
})

# ลบ comma ออกจากราคา
# เช่น "25,900.00" -> "25900.00"
df["unit_price"] = (
    df["unit_price"]
    .astype("string")
    .str.replace(",", "", regex=False)
)

# แปลงคอลัมน์ตัวเลขให้เป็น numeric
# ค่าที่แปลงไม่ได้ เช่น "not_available" จะกลายเป็น NaN
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["delivery_days"] = pd.to_numeric(
    df["delivery_days"],
    errors="coerce"
)

/tmp/ipykernel_5282/1869023569.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["quantity"] = df["quantity"].replace({
/tmp/ipykernel_5282/1869023569.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["unit_price"] = (
/tmp/ipykernel_5282/1869023569.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/in

# 6. จัดการค่าติดลบ


In [ ]:
# จำนวนวันจัดส่งติดลบไม่สมเหตุสมผล
# เปลี่ยนเป็นค่าว่างแทนการลบทั้งแถว
# ในไฟล์นี้ O1016 มี delivery_days = -2
df.loc[df["delivery_days"] < 0, "delivery_days"] = pd.NA

# 7. แทนหมวดหมู่สินค้าที่ไม่มีข้อมูล


In [ ]:
# หากไม่มี product_category ให้แทนด้วย "Unknown"
# เพื่อเก็บ order ไว้ใช้งานต่อ
# ในไฟล์นี้คือ O1011
df["product_category"] = df["product_category"].fillna("Unknown")

/tmp/ipykernel_5282/3058636612.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["product_category"] = df["product_category"].fillna("Unknown")


# 8. ตัดแถวที่ใช้ต่อไม่ได้


In [ ]:
# ตัดแถวที่ไม่มี customer_id
# เพราะไม่สามารถเชื่อมกับข้อมูลลูกค้าได้
# ในไฟล์นี้คือ O1012
df = df.dropna(subset=["customer_id"])

# ตัดแถวที่ไม่มีราคา
# เช่น unit_price เดิมเป็น "not_available"
# เพราะไม่สามารถคำนวณมูลค่าการขายได้
# ในไฟล์นี้คือ O1010
df = df.dropna(subset=["unit_price"])

# ตัดแถวที่ quantity ไม่มีค่า หรือ quantity <= 0
# ในไฟล์นี้ O1014 มี quantity = -3
df = df[
    df["quantity"].notna()
    & (df["quantity"] > 0)
]

# 9. ปรับชนิดข้อมูลหลัง clean


In [ ]:
# quantity ต้องเป็นจำนวนเต็ม และไม่มีค่าว่างแล้ว
df["quantity"] = df["quantity"].astype("int64")

# Int64 ตัว I ใหญ่ รองรับค่าว่างได้
# เหมาะกับ delivery_days เพราะบาง order ยังไม่มีข้อมูล
df["delivery_days"] = df["delivery_days"].astype("Int64")

# 10. ตรวจสอบผลลัพธ์


In [ ]:
print("จำนวนแถวก่อน clean:", len(raw_dirty_df))
print("จำนวนแถวหลัง clean:", len(df))

print("\nจำนวน null ในแต่ละคอลัมน์:")
print(df.isna().sum())

print("\nข้อมูลหลัง clean:")
display(df)

จำนวนแถวก่อน clean: 21
จำนวนแถวหลัง clean: 17

จำนวน null ในแต่ละคอลัมน์:
order_id            0
customer_id         0
order_date          1
product_category    0
quantity            0
unit_price          0
delivery_days       4
status              0
dtype: int64

ข้อมูลหลัง clean:


,order_id,customer_id,order_date,product_category,quantity,unit_price,delivery_days,status
0,O1001,C001,2026-01-03,Electronics,1,15900.0,3,Failed
1,O1002,C002,2026-01-05,Office Supplies,4,120.0,8,Failed
2,O1003,C003,2026-01-06,Furniture,1,8900.0,7,Succeeded
3,O1004,C001,2026-01-10,Office Supplies,2,350.0,10,Succeeded
4,O1005,C004,2026-01-12,Electronics,2,1290.0,5,Succeeded
5,O1006,C005,2026-01-14,Furniture,1,4500.0,<NA>,Failed
6,O1007,C006,2026-01-18,Office Supplies,10,45.0,2,Succeeded
8,O1008,C007,2026-01-21,Electronics,1,25900.0,12,Refunded
9,O1009,C008,2026-01-23,Furniture,2,3200.0,6,Succeeded
11,O1011,C003,2026-02-01,Unknown,5,85.0,<NA>,Refunded


# 11. บันทึกไฟล์


In [ ]:
# บันทึกวันที่เป็นรูปแบบ YYYY-MM-DD
# index=False ป้องกันไม่ให้ pandas เพิ่มคอลัมน์เลขลำดับแถวลงใน CSV
df.to_csv(
    "clean_orders.csv",
    index=False,
    date_format="%Y-%m-%d"
)